# Jedna partycja per device
Wszystkie dane urządzenia trafiają do jednej partycji bez clusteringu, w efekcie TYLKO jeden wiersz z ostatnią wartością!
Partition key = device_id 

In [1]:
import uuid
from cassandra.cluster import Cluster

In [2]:
# Otwieranie połączenia
cluster = Cluster(['cassandra_nosql_lab'], port=9042)
session = cluster.connect()
print("Połączono z Cassandra")

Połączono z Cassandra


In [3]:
# Tworzenie Keyspace
session.execute("""
    CREATE KEYSPACE IF NOT EXISTS lab_keyspace
    WITH replication = {'class': 'SimpleStrategy', 'replication_factor': '1'}
""")
print("Utworzono keyspace")

Utworzono keyspace


In [4]:
# Użycie Keyspace
session.set_keyspace('lab_keyspace')

# Tworzenie tabeli
session.execute("""
    CREATE TABLE IF NOT EXISTS events_by_device_1 (
        device_id TEXT,
        day TEXT,
        event_time TIMESTAMP,
        temperature FLOAT,
        humidity FLOAT,
        PRIMARY KEY (device_id)
    )
""")
print("Utworzono tabelę 'users'")

Utworzono tabelę 'users'


## Generate test data

In [5]:
!python ../_data_generator/main.py \
  --loader cassandra \
  --table events_by_device_1 \
  --records 1000 \
  --batch-size 100

Done. Loader=cassandra, Records=1000


In [6]:
rows = session.execute("""
SELECT count(*)
FROM
    events_by_device_1
""")
print("Znalezione rekordy:")
for row in rows:
    print(f"COUNT: {row.count}")

Znalezione rekordy:
COUNT: 10


In [7]:
rows = session.execute("""
SELECT *
FROM
    events_by_device_1
WHERE
    device_id='device_2'
LIMIT 10
""")
print("Znalezione rekordy:")
for row in rows:
    print(f"- {row.device_id} {row.day} ({row.temperature}°C, {row.humidity}%) – {row.event_time}")

Znalezione rekordy:
- device_2 2026-03-28 (25.90999984741211°C, 56.68000030517578%) – 2026-03-28 09:23:46.921000


In [8]:
# Zamknięcie połączenia
cluster.shutdown()